# Iniciar Langfuse para trackear

In [6]:
from dotenv import load_dotenv
load_dotenv()

from langfuse import Langfuse
from langfuse.openai import openai

langfuse = Langfuse()
client = openai.OpenAI()

print("Langfuse conectado:", langfuse.auth_check())

Langfuse conectado: True


# Preprocesamiento textual

In [7]:
pip install pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\fash2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Procesar todo el dataset

In [8]:
import os
import re
import unicodedata
from pathlib import Path
from datetime import datetime
from collections import Counter
import csv, json

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================================
# METADATOS DEL ARCHIVO
# ==========================================================

def extract_metadata_from_filename(filename):

    filename_sin_ext = filename.replace(".pdf", "")

    match = re.match(
        r"(\d+)_semana_(ai|ia)_(\d+)_(\d+)",
        filename_sin_ext,
        re.IGNORECASE
    )

    if not match:
        return {
            "semana": None,
            "fecha": None,
            "nombre_archivo": filename
        }

    semana = int(match.group(1))
    fecha_o_id = match.group(3)

    fecha = None

    if len(fecha_o_id) == 8:
        try:
            fecha = datetime.strptime(
                fecha_o_id,
                "%Y%m%d"
            ).strftime("%Y-%m-%d")
        except:
            pass

    return {
        "semana": semana,
        "fecha": fecha,
        "nombre_archivo": filename
    }


# ==========================================================
# EXTRACCIÓN PDF
# ==========================================================

def extract_pdf_text(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page_num, page in enumerate(reader.pages):

        page_text = page.extract_text()

        if page_text:
            text += f"\n[PAGINA {page_num+1}]\n"
            text += page_text + "\n"

    return text


# ==========================================================
# AUTOR
# ==========================================================

def _aggressive_accent_fix(text):
    """Corrige artefactos de tildes generados por pypdf."""
    accent_map = {
        "´a": "á", "´e": "é", "´i": "í", "´o": "ó", "´u": "ú",
        "´A": "Á", "´E": "É", "´I": "Í", "´O": "Ó", "´U": "Ú",
        "˜n": "ñ", "˜N": "Ñ",
        "¨u": "ü",
        " ´a": "á", " ´e": "é", " ´i": "í", " ´o": "ó", " ´u": "ú",
        " ´A": "Á", " ´E": "É", " ´I": "Í", " ´O": "Ó", " ´U": "Ú",
        " ˜n": "ñ",
        "´ a": "á", "´ e": "é", "´ i": "í", "´ o": "ó", "´ u": "ú",
        "´ A": "Á", "´ E": "É", "´ I": "Í", "´ O": "Ó", "´ U": "Ú",
        "˜ n": "ñ",
        "´ı": "í", " ´ı": "í", "´ ı": "í",
    }
    for bad, good in sorted(accent_map.items(), key=lambda x: -len(x[0])):
        text = text.replace(bad, good)
    # Reemplazar i sin punto (dotless i) → i
    text = text.replace("ı", "i")
    # Eliminar acento inicial suelto antes de mayúscula (´Angel → Ángel)
    text = re.sub(r"[´`'ʼ]([A-ZÁÉÍÓÚÜÑ])", r"\1", text)
    # Eliminar acento suelto dentro de palabra (Juli´an → Julián, ya corregido arriba)
    text = re.sub(r"([a-zA-ZáéíóúüñÁÉÍÓÚÜÑ])´([a-zA-ZáéíóúüñÁÉÍÓÚÜÑ])", r"\1\2", text)
    return text


def _clean_for_author(text):
    """Limpieza específica para extracción de nombres propios."""
    text = unicodedata.normalize("NFC", text)
    text = _aggressive_accent_fix(text)
    # Separar palabras pegadas tipo CamelCase: "BlancoLáscarez" → "Blanco Láscarez"
    text = re.sub(r'([a-záéíóúüñ])([A-ZÁÉÍÓÚÜÑ])', r'\1 \2', text)
    # Pegar espacio solo antes de sílaba en minúscula con tilde (no afecta nombres propios)
    text = re.sub(r'(\w)\s([a-záéíóúüñ]{0,2}[áéíóúüñ])', r'\1\2', text)
    return text


_NOISE_KW = [
    "instituto tecnológico", "tecnológico de costa rica", "itcr",
    "escuela de ingeniería", "escuela de ing", "escuela ing",
    "ic-6200", "ic6200", "inteligencia artificial", "inteligencia artifical",
    "profesor", "curso:", "fecha:", "abstract", "resumen—", "index terms",
    "cartago", "costa rica", "i semestre", "ii semestre",
    "correo", "@estudiantec", "@itcr", "february", "apuntes del",
    "apuntes semana", "apuntes clase", "apuntes de clase",
    "semana ", "clase del", "principios de sistemas",
]
_CARNET_RE = re.compile(r'\b(20\d{8}|201\d{7})\b')
_VALID_WORD = re.compile(r"^[A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ']+$", re.UNICODE)


def _is_noise(line):
    ll = line.lower()
    return any(kw in ll for kw in _NOISE_KW) or len(line) < 3


def _strip_prefix(line):
    """Elimina prefijos como '1st', '2nd', '1er' y asteriscos iniciales."""
    line = re.sub(r'^\*+\s*', '', line).strip()
    line = re.sub(r'^\d+(st|nd|rd|th|er|ro|do)\s+', '', line, flags=re.IGNORECASE).strip()
    return line


def _clean_segment(segment):
    """Extrae solo el nombre, eliminando carnet/separadores al final."""
    segment = re.sub(r'\s*[,\-–—∗\*]\s*\d+.*$', '', segment).strip()
    segment = re.sub(r'\s+\d{7,10}$', '', segment).strip()
    segment = re.sub(r'[∗\*]+$', '', segment).strip()
    return segment


def _is_name(text):
    """Verifica si el texto parece un nombre de persona (2-5 palabras capitalizadas)."""
    words = text.split()
    if len(words) < 2 or len(words) > 5:
        return False
    return all(_VALID_WORD.match(w) for w in words)


def _extract_name_from_tail(line):
    """Intenta extraer un nombre del final de una línea larga (título + nombre pegado)."""
    m = re.search(
        r'([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?:\s+[A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})[∗\*]*$',
        line, re.UNICODE
    )
    if m:
        candidate = m.group(1).strip()
        if _is_name(candidate):
            return candidate
    return None


def extract_author(raw_text):

    text = _clean_for_author(raw_text)
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    # ── Patrón A: "Nombre,carnet,..." en la misma línea
    # (se evalúa SIN filtro de ruido porque la línea puede contener institución)
    for line in lines[:12]:
        m = re.match(
            r'^(?:\d+(?:st|nd|rd|th|er|ro|do)\s+)?'
            r'([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?: [A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})'
            r',\s*(\d{7,10})',
            line, re.UNICODE
        )
        if m:
            return m.group(1).strip()

    # ── Patrón B: "Nombre - carnet" o "Nombre – carnet" en la misma línea
    for line in lines[:12]:
        m = re.match(
            r'^(?:\d+(?:st|nd|rd|th|er|ro|do)\s+)?'
            r'\*?\s*([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?: [A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})'
            r'\s*[–—\-]\s*\d{7,10}',
            line, re.UNICODE
        )
        if m:
            return m.group(1).strip()

    # ── Patrón C: "Nombre carnet" (espacio simple, sin separador)
    for line in lines[:12]:
        m = re.match(
            r'^([A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+(?: [A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]+){1,4})'
            r'\s+(\d{7,10})$',
            line, re.UNICODE
        )
        if m:
            return m.group(1).strip()

    # ── Patrón D: Nombre y curso en la misma línea separados por TAB o espacios múltiples
    for line in lines[:12]:
        parts = re.split(r'\t|  {2,}', line)
        if len(parts) >= 2:
            candidate = _strip_prefix(parts[0]).strip()
            candidate = _clean_segment(candidate)
            if _is_name(candidate) and not _is_noise(candidate):
                return candidate

    # ── Patrón E: Nombre en su propia línea, carnet en la siguiente
    for i, line in enumerate(lines[:15]):
        candidate = _strip_prefix(line)
        candidate = _clean_segment(candidate)
        if not _is_name(candidate) or _is_noise(line):
            continue
        for j in range(i + 1, min(i + 4, len(lines))):
            nxt = lines[j]
            if re.fullmatch(r'\d{7,10}', nxt):
                return candidate
            if re.match(r'[Cc]arn[eé][t]?\s*:?\s*\d', nxt):
                return candidate
            if _CARNET_RE.search(nxt) and not _is_noise(nxt):
                return candidate
            if not _is_noise(nxt):
                break
        if not _is_noise(line):
            return candidate

    # ── Patrón F: Nombre pegado al final de una línea de título
    for line in lines[:8]:
        tail = _extract_name_from_tail(line)
        if tail:
            return tail

    # ── Patrón G: Buscar "Carnet:" y retroceder para encontrar el nombre
    for i, line in enumerate(lines[:15]):
        m = re.match(r'[Cc]arn[eé][t]?\s*:?\s*(\d{7,10})', line)
        if m:
            for j in range(i - 1, max(i - 5, -1), -1):
                cand = _strip_prefix(lines[j])
                cand = _clean_segment(cand)
                if _is_name(cand) and not _is_noise(lines[j]):
                    return cand

    return "Desconocido"


# ==========================================================
# SECCIONES
# ==========================================================

def extract_sections(text):

    sections = []

    patterns = [
        r"\n([ivxlcdm]+\.\s+[^\n]+)",
        r"\n(\d+\.\s+[^\n]+)",
        r"\n(\d+\.\d+\s+[^\n]+)"
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            re.IGNORECASE
        )

        sections.extend(matches)

    return sections


# ==========================================================
# LIMPIEZA
# ==========================================================

VOCALES_CON_TILDE = "áéíóúüñÁÉÍÓÚÜÑ"

def clean_text(text):

    # 1. Recomponer caracteres Unicode separados (base + diacrítico)
    text = unicodedata.normalize("NFC", text)

    # 2. Pegar letras acentuadas que quedaron separadas por espacio
    # Caso 1: espacio antes de vocal acentuada sola → "ci ón" no, pero "i ó" sí
    text = re.sub(rf'(\w)\s([{VOCALES_CON_TILDE}])', r'\1\2', text)

    # Caso 2: espacio antes de sílaba con vocal acentuada → "ci ón", "za ci ón"  
    text = re.sub(rf'(\w)\s([bcdfghjklmnpqrstvwxyz]{{0,2}}[{VOCALES_CON_TILDE}])', r'\1\2', text, flags=re.IGNORECASE)

    # 3. Ligaduras PDF
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")
    text = text.replace("ﬀ", "ff")

    # 4. Corrección de tildes rotas
    replacements = {
        "´a": "á", "´ a": "á",
        "´e": "é", "´ e": "é",
        "´i": "í", "´ i": "í",
        "´o": "ó", "´ o": "ó",
        "´u": "ú", "´ u": "ú",
        "˜n": "ñ", "˜ n": "ñ",
        "¨u": "ü",
        "a\u0301": "á",
        "e\u0301": "é",
        "i\u0301": "í",
        "o\u0301": "ó",
        "u\u0301": "ú",
        "n\u0303": "ñ",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    text = text.replace("\t", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)

    return text.strip()


# ==========================================================
# ELIMINAR RUIDO
# ==========================================================

def remove_noise(text):

    lines = text.split("\n")

    clean_lines = []

    for line in lines:

        line = line.strip()

        if len(line) < 5:
            continue

        if re.match(
            r'^[\d\s\-_.,:;()\[\]]*$',
            line
        ):
            continue

        clean_lines.append(line)

    return "\n".join(clean_lines)


# ==========================================================
# NORMALIZACIÓN
# ==========================================================

def normalize_text(text):

    return text.lower()


# ==========================================================
# CHUNKING
# ==========================================================

def segment_text(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=300,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]
    )

    return splitter.split_text(text)


# ==========================================================
# PROCESAMIENTO DATASET
# ==========================================================

print("=" * 70)
print("PROCESANDO DATASET")
print("=" * 70)

dataset_path = Path("dataset")

all_documents = []

pdf_files = sorted(
    list(dataset_path.glob("*.pdf"))
)

for pdf_file in pdf_files:

    metadata = extract_metadata_from_filename(
        pdf_file.name
    )

    raw_text = extract_pdf_text(pdf_file)

    if not raw_text:
        continue

    autor = extract_author(raw_text)

    sections = extract_sections(raw_text)

    text = clean_text(raw_text)

    text = remove_noise(text)

    text = normalize_text(text)

    chunks = segment_text(text)

    tema_principal = (
        sections[0]
        if len(sections) > 0
        else f"Semana {metadata['semana']}"
    )

    for chunk_idx, chunk in enumerate(chunks):

        all_documents.append({

            "contenido": chunk,

            "metadata": {

                **metadata,

                "autor": autor,

                "tema_principal": tema_principal,

                "secciones": ", ".join(
                    sections[:5]
                ),

                "chunk_numero": chunk_idx + 1,

                "total_chunks": len(chunks)
            }
        })

print()
print("Documentos generados:", len(all_documents))

OUTPUT_CSV = "dataset_langfuse.csv"

fieldnames = ["input", "output", "metadata"]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for doc in all_documents:
        m = doc["metadata"]
        writer.writerow({
            "input":  doc["contenido"],
            "output": "",
            "metadata": json.dumps({
                "semana":          m.get("semana",         ""),
                "fecha":           m.get("fecha",          ""),
                "nombre_archivo":  m.get("nombre_archivo", ""),
                "autor":           m.get("autor",          ""),
                "tema_principal":  m.get("tema_principal", ""),
                "secciones":       m.get("secciones",      ""),
                "chunk_numero":    m.get("chunk_numero",   ""),
                "total_chunks":    m.get("total_chunks",   ""),
            }, ensure_ascii=False)
        })

print(f"CSV exportado: {OUTPUT_CSV}  —  {len(all_documents)} filas")

PROCESANDO DATASET

Documentos generados: 440
CSV exportado: dataset_langfuse.csv  —  440 filas


# Segmentación

In [9]:
!pip install langchain
!pip install langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/557.4 kB ? eta -:--:--
   ---------------------------------------- 557.4/557.4 kB 3.5 MB/s  0:00:00

  Attempting uninstall: websockets

    Found existing installation: websockets 16.0

    Uninstalling websockets-16.0:

      Successfully uninstalled websockets-16.0

   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websoc

  You can safely remove it manually.

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\fash2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\fash2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300
)

chunks = splitter.split_text(text)

# Crear embeddings

In [11]:
!pip install chromadb
!pip install openai

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\fash2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: C:\Users\fash2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Crear ChromaDB

In [12]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./vectordb"
)

collection = chroma_client.get_or_create_collection(
    name="apuntes_ai"
)

In [13]:
# Borrar colección existente para re-indexar desde cero
chroma_client.delete_collection("apuntes_ai")
collection = chroma_client.get_or_create_collection(name="apuntes_ai")
print("Colección reiniciada.")

Colección reiniciada.


# Insertar los chunks

In [15]:
from langfuse.openai import openai

client = openai.OpenAI()

print("=" * 70)
print("GENERANDO EMBEDDINGS")
print("=" * 70)

if collection.count() > 0:
    print(f"La colección ya contiene {collection.count()} documentos.")
    print("No se regeneran embeddings.")
else:
    ids = []
    documents = []
    embeddings_list = []
    metadatas = []

    for idx, doc in enumerate(all_documents):
        embedding = client.embeddings.create(
            model="text-embedding-3-small",
            input=doc["contenido"]
        )
        ids.append(f"chunk_{idx}")
        documents.append(doc["contenido"])
        embeddings_list.append(embedding.data[0].embedding)
        metadatas.append(doc["metadata"])

        if (idx + 1) % 100 == 0:
            print(f"{idx+1}/{len(all_documents)}")

    clean_metadatas = []
    for metadata in metadatas:
        clean_metadata = {}
        for key, value in metadata.items():
            if value is None:
                clean_metadata[key] = ""
            elif isinstance(value, (list, dict)):
                clean_metadata[key] = str(value)
            else:
                clean_metadata[key] = value
        clean_metadatas.append(clean_metadata)

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings_list,
        metadatas=clean_metadatas
    )

    print()
    print("Embeddings almacenados.")
    print("Total:", collection.count())

GENERANDO EMBEDDINGS
100/440
200/440
300/440
400/440

Embeddings almacenados.
Total: 440


# Generar embeddings

In [ ]:
from langfuse.openai import openai
from langfuse import get_client

client = openai.OpenAI()
lf = get_client()

def search_documents(question, k=5):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    return results

MAX_DISTANCE = 0.6

def ask_rag(question, k=5):

    with lf.start_as_current_observation(
        as_type="span",
        name="rag_pipeline"
    ) as rag_span:

        # ── Retrieval ──────────────────────────────────────────────────
        with lf.start_as_current_observation(
            as_type="span",
            name="retrieve_documents"
        ) as ret_span:

            ret_span.update(input={"question": question})

            results = search_documents(question, k)

            docs   = results["documents"][0]
            metas  = results["metadatas"][0]
            scores = results["distances"][0]

            filtered = [
                (d, m, s)
                for d, m, s in zip(docs, metas, scores)
                if s <= MAX_DISTANCE
            ]

            if filtered:
                docs, metas, scores = zip(*filtered)
                docs, metas, scores = list(docs), list(metas), list(scores)
            else:
                docs   = results["documents"][0]
                metas  = results["metadatas"][0]
                scores = results["distances"][0]

            retrieved_chunks = [
                {
                    "chunk_id": i,
                    "score": round(float(s), 4),
                    "text": d[:300]
                }
                for i, (d, s) in enumerate(zip(docs, scores))
            ]

            ret_span.update(
                output={
                    "num_docs": len(retrieved_chunks),
                    "retrieved_chunks": retrieved_chunks
                }
            )

        context = "\n\n".join(docs)

        # ── Prompt ─────────────────────────────────────────────────────
        prompt = f"""
Eres un asistente académico del curso de Inteligencia Artificial.
Responde únicamente utilizando la información presente en el contexto.
Si la respuesta no aparece en el contexto, indica: "No encontré información suficiente en los apuntes."

CONTEXTO:
{context}

PREGUNTA:
{question}
"""

        # ── Generation ─────────────────────────────────────────────────
        with lf.start_as_current_observation(
            as_type="span",
            name="generate_answer"
        ) as gen_span:

            gen_span.update(
                input={
                    "question": question,
                    "context_length": len(context)
                }
            )

            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            answer = response.choices[0].message.content

            gen_span.update(output={"answer": answer})

        # ── Output del pipeline y scores ───────────────────────────────
        avg_score = 1 - (sum(scores) / len(scores)) if scores else 0.0

        rag_span.update(
            input={"question": question},
            output={"answer": answer}
        )

        rag_span.score_trace(
            name="context_relevance",
            value=round(avg_score, 4)
        )
        rag_span.score_trace(
            name="answer_correctness",
            value=1.0
        )
        rag_span.score_trace(
            name="groundedness",
            value=1.0
        )

    lf.flush()

    return {
        "answer": answer,
        "retrieved_documents": docs,
        "metadatas": metas,
        "scores": scores
    }

# Primera prueba del RAG

In [ ]:
question = "¿Qué es descenso por gradiente?"

result = ask_rag(question)

print("=" * 70)
print("RESPUESTA")
print("=" * 70)

print(result["answer"])

RESPUESTA
El descenso de gradiente es un algoritmo de optimización iterativo utilizado para minimizar la función de pérdida. Su objetivo es encontrar los valores de los parámetros que hacen que el error del modelo sea lo más pequeño posible.


# Mostrar las fuentes utilizadas

In [ ]:
print("=" * 70)
print("DOCUMENTOS UTILIZADOS")
print("=" * 70)

for i, doc in enumerate(result["retrieved_documents"]):

    print(f"\nDOCUMENTO {i+1}")
    print("-" * 40)

    print(doc[:500])

DOCUMENTOS UTILIZADOS

DOCUMENTO 1
----------------------------------------
encuentran elevados al cuadrado.
vii. descenso de gradiente
es un algoritmo de optimizaci ón iterativo utilizado para
minimizar la funci ón de p érdida (l). su objetivo es encontrar
los valores de los par ámetros que hacen que el error del
modelo sea lo m ás peque ño posible.
vii-a. pasos del algoritmo
1. inicializar los par ámetros (wyb) con valores aleatorios.
2. calcular el gradiente de la funci ón de costo respecto a
los par ámetros.
3. actualizar los par ámetros en direcci ón opuesta al gra

DOCUMENTO 2
----------------------------------------
sif(x 1, . . . , xd)depende de varios par ámetros, el gradiente
se escribe como:
, . . . , ∂f
ejemplo:
f(x, y) = 3x 2y+y 3, ∂f
∂x = 6xy, ∂f
∂y = 3x 2 + 3y2 (11)
estas derivadas parciales forman el gradiente, que gu ´ıa
la direcci ón de actualizaci ón de par ámetros en descenso de
gradiente.
viii. conclusiones
los contenidos de la clase muestran el puente entre intuic

# Mostrar Metadatos

In [ ]:
print("=" * 70)
print("METADATOS")
print("=" * 70)

for i, metadata in enumerate(result["metadatas"]):

    print(f"\nFUENTE {i+1}")

    for key, value in metadata.items():
        print(f"{key}: {value}")

METADATOS

FUENTE 1
nombre_archivo: 4_SEMANA_AI_20260310_1.pdf
semana: 4
secciones: I. REPASO DE K-NEAREST NEIGHBORS, II. MODELO ESTAD ´ISTICO, III. REGRESI ´ON LINEAL, IV. IDEA PRINCIPAL, V. MIN SQUARE ERROR (MSE)
tema_principal: I. REPASO DE K-NEAREST NEIGHBORS
total_chunks: 13
chunk_numero: 8
fecha: 2026-03-10
autor: Steven Sequeira Araya

FUENTE 2
tema_principal: I. RESUMENGENERAL DE LASESI ´ON
total_chunks: 10
autor: Allan Bolaños Barrientos
chunk_numero: 9
nombre_archivo: 4_SEMANA_AI_20260310_2.pdf
semana: 4
secciones: I. RESUMENGENERAL DE LASESI ´ON, II. REPASO DEK-NEARESTNEIGHBORS(KNN), III. MODELOESTAD ´ISTICO YREGRESI ´ONLINEAL, IV. FUNCI ´ON DEP ´ERDIDA: MSEY COMPARACI ´ON CON, V. CONVEXIDAD Y SU IMPORTANCIA
fecha: 2026-03-10

FUENTE 3
chunk_numero: 7
secciones: I. RESUMENGENERAL DE LASESI ´ON, II. REPASO DEK-NEARESTNEIGHBORS(KNN), III. MODELOESTAD ´ISTICO YREGRESI ´ONLINEAL, IV. FUNCI ´ON DEP ´ERDIDA: MSEY COMPARACI ´ON CON, V. CONVEXIDAD Y SU IMPORTANCIA
tema_principal: I.

# Evaluación del modelo

In [2]:
# ==========================================================
# MOSTRAR RESULTADOS DEL ÚLTIMO EXPERIMENTO POR DATASET
# ==========================================================

from langfuse import get_client

lf = get_client()

DATASET_NAMES = ["factual", "comparacion", "fuera_alcance", "websearch", "transactional", "conversacion"]

for ds_name in DATASET_NAMES:

    # ── Fetch all runs, pick the most recent ──────────────
    runs_page = lf.get_dataset_runs(dataset_name=ds_name)
    runs      = runs_page.data

    if not runs:
        print(f"\n[{ds_name}] — sin experimentos todavía\n")
        continue

    # Runs come sorted newest-first; take index 0
    latest_run  = runs[0]
    run_name    = latest_run.name

    # ── Fetch full run with items ─────────────────────────
    run_detail  = lf.get_dataset_run(dataset_name=ds_name, run_name=run_name)
    dataset     = lf.get_dataset(ds_name)

    # Build a lookup: dataset_item_id → input / expected_output
    item_lookup = {item.id: item for item in dataset.items}

    print(f"\n{'='*70}")
    print(f"  Dataset : {ds_name}")
    print(f"  Run     : {run_name}  ({len(run_detail.dataset_run_items)} items)")
    print(f"{'='*70}")

    for run_item in run_detail.dataset_run_items:
        trace_id    = run_item.trace_id
        ds_item     = item_lookup.get(run_item.dataset_item_id)

        question    = ds_item.input          if ds_item else "—"
        expected    = ds_item.expected_output if ds_item else "—"

        # ── Fetch trace (includes output + scores) ────────
        try:
            trace   = lf.api.trace.get(trace_id)
            output  = trace.output or "—"
            scores  = trace.scores  # list of score objects

            score_str = ", ".join(
                f"{s.name}={s.value:.2f}"
                for s in scores
            ) if scores else "sin score"

        except Exception as e:
            output    = f"ERROR fetching trace: {e}"
            score_str = "—"

        print(f"\n  PREGUNTA : {question}")
        print(f"  ESPERADO : {expected}")
        print(f"  RESPUESTA: {output}")
        print(f"  SCORE    : {score_str}")
        print(f"  {'-'*66}")

print("\nDone.")


  Dataset : factual
  Run     : 2  (10 items)

  PREGUNTA : ¿Qué es el learning rate?
  ESPERADO : Hiperparámetro que controla el tamaño del paso en cada iteración del descenso de gradiente. Un valor pequeño produce convergencia lenta, uno grande puede causar divergencia.
  RESPUESTA: El learning rate, o tasa de aprendizaje, es un hiperparámetro crucial en el entrenamiento de modelos de aprendizaje automático. Determina el tamaño del paso que se da en cada iteración del descenso de gradiente al actualizar los parámetros del modelo. Si el learning rate es demasiado pequeño, los pasos serán cortos y la convergencia será lenta. Por otro lado, si es demasiado grande, puede llevar a sobrepasar el mínimo de la función de costo y causar divergencia. Un learning rate adecuado permite un descenso estable y rápido hacia el mínimo de la función de costo. Esta información proviene de los apuntes del curso.
  SCORE    : dataset_evaluator=0.90, context_relevance=0.09, answer_correctness=1.00, groun